# Results figures

Reads `results/metrics/results.json` + `results/embeddings_test.npz` (produced by
`python -m ids_anomaly.cli run`) and generates every figure embedded in `docs/results.md` and
the root README. No modeling logic here -- purely plotting already-computed results.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
ASSETS = ROOT / "docs" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

with open(ROOT / "results" / "metrics" / "results.json") as f:
    metrics = json.load(f)
data = np.load(ROOT / "results" / "embeddings_test.npz", allow_pickle=True)
metrics.keys()

## PCA cumulative explained variance

In [ ]:
curve = metrics["pca"]["cumulative_explained_variance"]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(curve) + 1), curve, marker="o")
ax.axvline(3, color="crimson", linestyle="--", label="components used downstream (3)")
ax.set_xlabel("n components")
ax.set_ylabel("cumulative explained variance")
ax.legend()
fig.tight_layout()
fig.savefig(ASSETS / "pca_explained_variance.png", dpi=150)

## Anomaly detector comparison (ROC-AUC / PR-AUC)

In [ ]:
ad = metrics["anomaly_detection"]
df = pd.DataFrame(ad).T[["roc_auc", "pr_auc", "f1"]].astype(float)
fig, ax = plt.subplots(figsize=(8, 4))
df.plot.bar(ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Anomaly detector comparison (test split)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ASSETS / "anomaly_detector_comparison.png", dpi=150)

## Per-category detection rate heatmap

In [ ]:
rows = []
for method, records in metrics["per_category_detection_rate"].items():
    for r in records:
        rows.append({"method": method, **r})
cat_df = pd.DataFrame(rows)
pivot = cat_df[cat_df["attack_category"] != "normal"].pivot(index="method", columns="attack_category", values="detection_rate")
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1, ax=ax)
ax.set_title("Detection rate by attack category (top-20% threshold)")
fig.tight_layout()
fig.savefig(ASSETS / "per_category_detection_rate.png", dpi=150)

## Clustering quality by embedding

In [ ]:
rows = []
for embed, methods in metrics["clustering"].items():
    for method, m in methods.items():
        rows.append({"embedding": embed, "method": method, "ari": m["ari"], "nmi": m["nmi"]})
clus_df = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 4))
clus_pivot = clus_df.pivot(index="method", columns="embedding", values="nmi")
clus_pivot.plot.bar(ax=ax)
ax.set_ylabel("NMI vs. attack_category")
ax.set_title("Clustering quality by embedding")
fig.tight_layout()
fig.savefig(ASSETS / "clustering_by_embedding.png", dpi=150)

## Static 3D scatter (a fixed reference view -- the dashboard is the real interactive one)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

emb = data["umap"]
colors = pd.Series(data["attack_category"].astype(str)).map(
    {"normal": "#3288bd", "dos": "#d53e4f", "probe": "#fdae61", "r2l": "#66c2a5", "u2r": "#9e0142"}
).fillna("#999999")
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(emb[:, 0], emb[:, 1], emb[:, 2], c=colors, s=4, alpha=0.5)
ax.set_title("UMAP embedding of NSL-KDD test flows, colored by attack category")
fig.tight_layout()
fig.savefig(ASSETS / "umap_3d_static.png", dpi=150)